In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from IPython.display import Markdown

In [2]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Convert sentiment category name

In [3]:
# Map labels to integers
sentiment_categories=['負面','正面']

In [4]:

sentimentlabel_to_id = { cate : i for i, cate in enumerate(sentiment_categories)}

In [5]:
sentimentlabel_to_id

{'負面': 0, '正面': 1}

In [6]:
id_to_sentimentlabel = { i : cate for i, cate in enumerate(sentiment_categories)}

In [7]:
id_to_sentimentlabel

{0: '負面', 1: '正面'}

## Convert news category name ('政治','科技','運動',...) into number (0,1,2,...)

In [8]:
# Map labels to integers
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']

In [9]:

newslabel_to_id = { cate : i for i, cate in enumerate(news_categories)}

In [10]:
newslabel_to_id

{'政治': 0,
 '科技': 1,
 '運動': 2,
 '證卷': 3,
 '產經': 4,
 '娛樂': 5,
 '生活': 6,
 '國際': 7,
 '社會': 8,
 '文化': 9,
 '兩岸': 10}

In [11]:
id_to_newslabel = { i : cate for i, cate in enumerate(news_categories)}

In [12]:
id_to_newslabel

{0: '政治',
 1: '科技',
 2: '運動',
 3: '證卷',
 4: '產經',
 5: '娛樂',
 6: '生活',
 7: '國際',
 8: '社會',
 9: '文化',
 10: '兩岸'}

In [13]:
from custom_qwen_model import QwenForClassifier

In [14]:
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [15]:

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [16]:
# 在外部先載入base_model預訓練權重(不包含分類層)
full_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
#
hidden_size = full_model.config.hidden_size
model_sentiment_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels= len(sentiment_categories))

# 移動到指定設備
model_sentiment_classifier = model_sentiment_classifier.to(device)

model_path_sentiment = "trained_sentiment_classifier_5epochs-acc0.93"
model_sentiment_classifier.load_model(model_path_sentiment, device=device)

已載入分類器權重: trained_sentiment_classifier_5epochs-acc0.93\classifier_weights.pt


True

In [17]:
hidden_size = full_model.config.hidden_size
model_news_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels= len(news_categories))

# 移動到指定設備
model_news_classifier = model_news_classifier.to(device)

model_path_news = "trained_news_classifier_5epochs-acc0.90"
model_news_classifier.load_model(model_path_news, device=device)

已載入分類器權重: trained_news_classifier_5epochs-acc0.90\classifier_weights.pt


True

In [18]:
model_sentiment_classifier

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMS

In [19]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model_sentiment_classifier(**inputs)
    probs = torch.softmax(outputs['logits'], dim=1)
    prediction = id_to_sentimentlabel[ probs.argmax().item() ]
    confidence = probs.max().item()
    return prediction, confidence

In [20]:
def predict_news_category(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model_news_classifier(**inputs)
    probs = torch.softmax(outputs['logits'], dim=1)
    prediction = id_to_newslabel[ probs.argmax().item() ]
    confidence = probs.max().item()
    return prediction, confidence

In [21]:

# Function to make predictions
def predict_sentiment(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_sentiment_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits
    
    
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_sentimentlabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_sentimentlabel[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }

In [22]:

# Function to make predictions
def predict_news_category(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_news_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits
    
    
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_newslabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_newslabel[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [23]:
%%time
input_text="我覺得這個產品很好用"
predict_sentiment(input_text)


CPU times: total: 188 ms
Wall time: 183 ms


{'text': '我覺得這個產品很好用',
 'classification': '正面',
 'confidence': 0.97,
 'probabilities': {'負面': 0.03, '正面': 0.97}}

In [24]:
%%time
input_text="我不覺得這個產品很好用"
predict_sentiment(input_text)


CPU times: total: 15.6 ms
Wall time: 19.3 ms


{'text': '我不覺得這個產品很好用',
 'classification': '負面',
 'confidence': 0.95,
 'probabilities': {'負面': 0.95, '正面': 0.05}}

In [25]:
%%time
input_text="民進黨與中國國民黨的關係"
predict_news_category(input_text)


CPU times: total: 15.6 ms
Wall time: 16 ms


{'text': '民進黨與中國國民黨的關係',
 'classification': '政治',
 'confidence': 0.9,
 'probabilities': {'政治': 0.9,
  '科技': 0.0,
  '運動': 0.0,
  '證卷': 0.01,
  '產經': 0.0,
  '娛樂': 0.0,
  '生活': 0.01,
  '國際': 0.02,
  '社會': 0.0,
  '文化': 0.01,
  '兩岸': 0.03}}

# Text generation fro full_model

In [26]:
def generate_text(input_prompt):
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [29]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

1. 均衡飲食：攝取均衡的營養，包括充足的碳水化合物、脂肪、蛋白质和维生素及矿物质。
2. 經常運動：定期進行有氧 exercise 和肌肉強化 exercise，有助於提升心肺功能和提高體重管理能力。
3. 保持良好作息：確保充足的睡眠和良好的休息時間，並避免過度使用電子設備。這有助於保持身心平衡和身心健康。

In [30]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: total: 3.08 s
Wall time: 3.23 s


1. 調整能源使用：使用更節能的設備和能源，如LED燈光、低能耗的電器等。
2. 道路改善：建造或更新道路以提高其效率和流量，减少车辆尾气排放。
3. 氒氣排放控制：使用可回收材料制作垃圾袋、塑料瓶和其他物品，减少对环境的影响。
4. 培養低碳生活方式：例如少开车多走路、骑自行车或者步行，以及节约用水用电。
5. 限制工业生产：鼓励企业采用更环保的生产工艺和技术，减少废气、废水和废渣的产生。
6. 加强环境保护教育：提高公众对环境保护的认识和意识，鼓励人们采取积极行动来保护环境。
7. 推广绿色出行方式：鼓励更多人选择公共交通工具、骑行、步行等方式代替私家车出行。

In [31]:
text = "這個產品品質差，服務更糟糕。文本情緒是正面還是負面？"
result = generate_text(text)
Markdown(result)

根據你提供的文本信息，可以得出以下的結論：

- 文本中的“品質差”和“服務更糟糕”都是负面的表現。
- 這兩種描述都表明用戶對產品或服務的體驗感到失望或者不滿。

因此，根據這些詞句，可以確定該文本的情感狀態是**負面**。